In [ ]:
# --- Arranque del entorno local (en Google Colab no cambia nada) ---
import pathlib
import sys
import types

try:
    _raiz = next(
        d
        for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
        if (d / "curso_setup.py").exists()
    )
    sys.path.insert(0, str(_raiz))
    import curso_setup
except StopIteration:  # Google Colab: se usa un sustituto mínimo
    import subprocess

    def _clonar(destino="curso_IA_CHEC"):
        if not pathlib.Path(destino).is_dir():
            subprocess.run(
                ["git", "clone", "https://github.com/UN-GCPDS/curso_IA_CHEC.git", destino],
                check=True,
            )
        return pathlib.Path(destino)

    def _descargar(file_id, destino):
        if not pathlib.Path(destino).exists():
            import gdown

            gdown.download(id=file_id, output=destino, quiet=False)
        return pathlib.Path(destino)

    curso_setup = types.SimpleNamespace(
        en_colab=lambda: True,
        init=lambda *a, **k: pathlib.Path.cwd(),
        clonar_curso=_clonar,
        descargar_drive=_descargar,
    )

curso_setup.init()

# Búsqueda de Hiperparámetros en Modelos de Regresión

## Paso 1: Instalación y Carga de Librerías




In [ ]:
import os
import itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression, ElasticNet, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
import tensorflow as tf
from tensorflow.keras.datasets import boston_housing

In [ ]:
import warnings
from sklearn.exceptions import FitFailedWarning
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FitFailedWarning)
warnings.filterwarnings('ignore', message=".*The covariance matrix of class.*")
warnings.filterwarnings('ignore', message=".*shrinkage not supported with 'svd' solver.*")

## Paso 2: Carga de los Datos



In [ ]:
# Cargar datos
(Xtrain, ytrain), (Xtest, ytest) = boston_housing.load_data()
print(f'Xtrain shape: {Xtrain.shape}, ytrain shape: {ytrain.shape}')
print(f'Xtest shape: {Xtest.shape}, ytest shape: {ytest.shape}')

Xtrain shape: (404, 13), ytrain shape: (404,)
Xtest shape: (102, 13), ytest shape: (102,)


## Paso 3: Modelos de regresión

###ElasticNet

In [ ]:
elastic_net_model = ElasticNet(alpha=0.5, l1_ratio=0.5)
elastic_net_pipeline = Pipeline([('scaler', MinMaxScaler()), ('regressor', elastic_net_model)])
elastic_net_pipeline.fit(Xtrain, ytrain)
y_pred = elastic_net_pipeline.predict(Xtest)
mse = mean_squared_error(ytest, y_pred)
print(f'ElasticNet - MSE: {mse:.4f}')

ElasticNet - MSE: 57.6822


### SVR

In [ ]:
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr_pipeline = Pipeline([('scaler', MinMaxScaler()), ('regressor', svr_model)])
svr_pipeline.fit(Xtrain, ytrain)
y_pred = svr_pipeline.predict(Xtest)
mse = mean_squared_error(ytest, y_pred)
print(f'SVR - MSE: {mse:.4f}')

SVR - MSE: 31.5931


### RandomForest

In [ ]:
random_forest_model = RandomForestRegressor(n_estimators=50, max_depth=10)
random_forest_pipeline = Pipeline([('scaler', MinMaxScaler()), ('regressor', random_forest_model)])
random_forest_pipeline.fit(Xtrain, ytrain)
y_pred = random_forest_pipeline.predict(Xtest)
mse = mean_squared_error(ytest, y_pred)
print(f'RandomForest - MSE: {mse:.4f}')

RandomForest - MSE: 13.3805


### KernelRidge

In [ ]:
kernel_ridge_model = KernelRidge(alpha=1.0, kernel='linear')
kernel_ridge_pipeline = Pipeline([('scaler', MinMaxScaler()), ('regressor', kernel_ridge_model)])
kernel_ridge_pipeline.fit(Xtrain, ytrain)
y_pred = kernel_ridge_pipeline.predict(Xtest)
mse = mean_squared_error(ytest, y_pred)
print(f'KernelRidge - MSE: {mse:.4f}')

KernelRidge - MSE: 27.5564


### GaussianProcess

In [ ]:
gaussian_process_model = GaussianProcessRegressor()
gaussian_process_pipeline = Pipeline([('scaler', MinMaxScaler()), ('regressor', gaussian_process_model)])
gaussian_process_pipeline.fit(Xtrain, ytrain)
y_pred = gaussian_process_pipeline.predict(Xtest)
mse = mean_squared_error(ytest, y_pred)
print(f'GaussianProcess - MSE: {mse:.4f}')

GaussianProcess - MSE: 224.9709


## Búsqueda de hiperparámetros

In [ ]:
# Definir regresores y sus hiperparámetros
regressors = {
    'ElasticNet': (
        ElasticNet(),
        {   'regressor__alpha': [0.1, 0.5, 1.0, 10.0],
            'regressor__l1_ratio': [0.001, 0.1, 0.5, 0.9, 0.9999]}
    ),
    'RandomForest': (
        RandomForestRegressor(),
        {   'regressor__n_estimators': [10, 50, 100],
            'regressor__max_depth': [10, 20, 30, None],
            'regressor__min_samples_split': [2, 5]}
    ),
    'KernelRidge': (
        KernelRidge(),
        {   'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
            'regressor__kernel': ['linear', 'rbf'],
            'regressor__gamma': [None, 0.1, 1.0]  }
    ),
    'GaussianProcess': (
        GaussianProcessRegressor(),
        {   'regressor__kernel': [C(1.0) * RBF(length_scale=l) for l in [0.1, 1.0]],
            'regressor__alpha': [1e-10, 1e-5]}
    ),
    'SVR': (
        SVR(),
        {   'regressor__C': [0.1, 1.0, 10.0],
            'regressor__epsilon': [0.01, 0.1, 0.5]}
    )
}

# Definir escaladores a probar
scalers = {
    'None': None,
    'MinMaxScaler': MinMaxScaler(),
    'StandardScaler': StandardScaler()
}

# Evaluación de modelos
best_model = None
best_mse = float('inf')

for scaler_name, scaler in scalers.items():
    for reg_name, (regressor, param_grid) in regressors.items():
        print(f'Probando modelo: {reg_name} con escalador: {scaler_name}...')

        # Construcción del pipeline
        steps = [('regressor', regressor)]
        if scaler:
            steps.insert(0, ('scaler', scaler))

        pipeline = Pipeline(steps)

        # Ejecutar GridSearchCV
        grid_search = GridSearchCV(
            pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=0
        )

        grid_search.fit(Xtrain, ytrain)

        # Evaluar el mejor modelo encontrado
        y_pred = grid_search.best_estimator_.predict(Xtest)
        mse = mean_squared_error(ytest, y_pred)

        print(f'{reg_name} con {scaler_name}: Mejor MSE en validación = {mse:.4f}')

        # Guardar el mejor modelo
        if mse < best_mse:
            best_mse = mse
            best_model = (reg_name, scaler_name, grid_search.best_estimator_)

joblib.dump(best_model[2], f'{best_model[0]}_best_model_regresion.pkl')
print(f'\nEl mejor modelo es {best_model[0]} con escalador {best_model[1]}, MSE={best_mse:.4f}')

Probando modelo: ElasticNet con escalador: None...
ElasticNet con None: Mejor MSE en validación = 21.6522
Probando modelo: RandomForest con escalador: None...
RandomForest con None: Mejor MSE en validación = 13.4088
Probando modelo: KernelRidge con escalador: None...
KernelRidge con None: Mejor MSE en validación = 22.8898
Probando modelo: GaussianProcess con escalador: None...
GaussianProcess con None: Mejor MSE en validación = 115.1962
Probando modelo: SVR con escalador: None...
SVR con None: Mejor MSE en validación = 61.4918
Probando modelo: ElasticNet con escalador: MinMaxScaler...
ElasticNet con MinMaxScaler: Mejor MSE en validación = 22.1319
Probando modelo: RandomForest con escalador: MinMaxScaler...
RandomForest con MinMaxScaler: Mejor MSE en validación = 13.6870
Probando modelo: KernelRidge con escalador: MinMaxScaler...
KernelRidge con MinMaxScaler: Mejor MSE en validación = 14.5908
Probando modelo: GaussianProcess con escalador: MinMaxScaler...
GaussianProcess con MinMaxScale

# Búsqueda de Hiperparámetros en Modelos de Clasificación

## Paso 2: Carga de los Datos

In [ ]:
# Cargar el dataset MNIST desde TensorFlow
(X_train_full, y_train_full), (X_test_full, y_test_full) = tf.keras.datasets.mnist.load_data()

# Normalizar los datos (convertir los valores de los píxeles a [0,1])
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test_full = X_test_full.astype(np.float32) / 255.0

# Reducir el tamaño de los datos para una ejecución más rápida
sample_size = 200
X_train_full = X_train_full[:sample_size]
y_train_full = y_train_full[:sample_size]
X_test_full = X_test_full[:int(sample_size * 0.2)]
y_test_full = y_test_full[:int(sample_size * 0.2)]

# Partición de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

Xtrain = X_train.reshape(-1, 28 * 28)
ytrain = y_train
Xtest = X_test.reshape(-1, 28 * 28)
ytest = y_test

print(f'Tamaño reducido de datos - Xtrain: {Xtrain.shape}, ytrain: {ytrain.shape}')
print(f'Tamaño reducido de datos - Xtest: {Xtest.shape}, ytest: {ytest.shape}')

Tamaño reducido de datos - Xtrain: (160, 784), ytrain: (160,)
Tamaño reducido de datos - Xtest: (40, 784), ytest: (40,)


## Paso 3: Modelos de clasificación

### Quadratic Discriminant Analysis


In [ ]:
qda_model = QuadraticDiscriminantAnalysis()
qda_pipeline = Pipeline([('scaler', MinMaxScaler()), ('classifier', qda_model)])
qda_pipeline.fit(Xtrain, ytrain)
y_pred = qda_pipeline.predict(Xtest)
accuracy = accuracy_score(ytest, y_pred)
print(f'Quadratic Discriminant Analysis - Accuracy: {accuracy:.4f}')

Quadratic Discriminant Analysis - Accuracy: 0.2250


/usr/local/lib/python3.11/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/p

### Logistic Regression

In [ ]:
logistic_model = LogisticRegression(C=1.0)
logistic_pipeline = Pipeline([('scaler', MinMaxScaler()), ('classifier', logistic_model)])
logistic_pipeline.fit(Xtrain, ytrain)
y_pred = logistic_pipeline.predict(Xtest)
accuracy = accuracy_score(ytest, y_pred)
print(f'Logistic Regression - Accuracy: {accuracy:.4f}')

Logistic Regression - Accuracy: 0.8250


### K-Nearest Neighbors (KNN)

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5, weights='uniform', p=2)
knn_pipeline = Pipeline([('scaler', MinMaxScaler()), ('classifier', knn_model)])
knn_pipeline.fit(Xtrain, ytrain)
y_pred = knn_pipeline.predict(Xtest)
accuracy = accuracy_score(ytest, y_pred)
print(f'KNeighbors - Accuracy: {accuracy:.4f}')

KNeighbors - Accuracy: 0.7750


### Decision Tree Model

In [ ]:
decision_tree_model = DecisionTreeClassifier(max_depth=10, min_samples_split=2)
decision_tree_pipeline = Pipeline([('scaler', MinMaxScaler()), ('classifier', decision_tree_model)])
decision_tree_pipeline.fit(Xtrain, ytrain)
y_pred = decision_tree_pipeline.predict(Xtest)
accuracy = accuracy_score(ytest, y_pred)
print(f'Decision Tree - Accuracy: {accuracy:.4f}')

Decision Tree - Accuracy: 0.6250


### Support Vector Classifier (SVC)

In [ ]:
svc_model = SVC(C=1.0, kernel='rbf', gamma=0.001)
svc_pipeline = Pipeline([('scaler', MinMaxScaler()), ('classifier', svc_model)])

svc_pipeline.fit(Xtrain, ytrain)
y_pred = svc_pipeline.predict(Xtest)
accuracy = accuracy_score(ytest, y_pred)
print(f'SVC - Accuracy: {accuracy:.4f}')

SVC - Accuracy: 0.3000


## Búsqueda de hiperparámetros



In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import joblib


# Definir los modelos y transformaciones en un pipeline
steps = [
    [('rep', PCA()), ('cla', GaussianNB())],
    [('rep', PCA()), ('cla', SGDClassifier())],
    [('rep', PCA()), ('cla', LinearDiscriminantAnalysis())],
    [('rep', PCA()), ('cla', QuadraticDiscriminantAnalysis())],
    [('rep', PCA()), ('cla', KNeighborsClassifier())],
    [('rep', PCA()), ('cla', LogisticRegression())],
    [('cla', SVC())],
    [('rep', PCA()), ('cla', RandomForestClassifier())]
]

# Definición de hiperparámetros para cada modelo
parameters = [
    {
        'rep__n_components': [0.8, 0.9]
    },
    {
        'rep__n_components': [0.8, 0.9],
        'cla__loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron'],
        'cla__penalty': ['l2', 'l1', 'elasticnet'],
        'cla__alpha': [1e-6, 1, 10]
    },
    {
        'rep__n_components': [0.8, 0.9],
        'cla__solver': ['svd', 'lsqr', 'eigen'],
        'cla__shrinkage': [None, 'auto', 1e-1, 1]
    },
    {
        'rep__n_components': [0.8, 0.9],
        'cla__reg_param': [0.0, 1.0e-1, 1.0e-3]
    },
    {
        'rep__n_components': [0.8, 0.9],
        'cla__n_neighbors': [1, 5, 10, 100],
        'cla__weights': ['uniform', 'distance'],
        'cla__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'cla__p': [2, 1, 3]
    },
    {
        'cla__penalty': ['l1', 'l2', 'elasticnet', 'none'],
        'cla__solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
    },
    {
        'cla__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'cla__degree': [3, 4, 10]
    },
    {
        'rep__n_components': [0.8, 0.9],
        'cla__n_estimators': [50, 100, 500],
        'cla__max_features': ['sqrt', 'log2', None],
        'cla__max_depth': [10, 50, 100, None]
    },
    {
        'rep__n_components' : [0.8,0.9],
        'cla__tipo':['NAIVE','QDA',"LDAISO","LDAANISO"]
    },
    {
        'cla__tipo':['NAIVE','QDA',"LDAISO","LDAANISO"]
    }

]


# Etiquetas de los modelos
label_models = [
    'GAUSSIANNB_PCA', 'SGD_PCA', 'LDA_PCA', 'QDA_PCA',
    'KNEIGHBORS_PCA', 'LOGREG_PCA', 'SVC', 'RANDOMFOREST_PCA'
]

best_score = -np.inf
best_models = []
filename = 'best_models'

for i in range(len(steps)):
    print(f'Entrenando modelo: {label_models[i]} ({i+1}/{len(steps)})...')

    grid_search = GridSearchCV(
        Pipeline(steps[i]),
        parameters[i],
        n_jobs=-1,
        cv=5,
        scoring='balanced_accuracy',
        verbose=10
    )

    grid_search.fit(Xtrain, ytrain)
    current_score = grid_search.best_score_

    print(f'Mejor modelo para {label_models[i]}: {grid_search.best_params_} con puntaje {current_score:.4f}')

    # Guardar el mejor modelo si su puntaje es mayor
    if current_score > best_score:
        best_score = current_score
        best_model = grid_search.best_estimator_
        best_model_name = label_models[i]


joblib.dump(best_model, f'{best_model_name}_best_model_clasificacion.pkl')
print(f'\nEl mejor modelo guardado es {best_model_name} con una puntuación de {best_score:.4f}')

Entrenando modelo: GAUSSIANNB_PCA (1/8)...
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Mejor modelo para GAUSSIANNB_PCA: {'rep__n_components': 0.8} con puntaje 0.8000
Entrenando modelo: SGD_PCA (2/8)...
Fitting 5 folds for each of 90 candidates, totalling 450 fits
Mejor modelo para SGD_PCA: {'cla__alpha': 1e-06, 'cla__loss': 'squared_hinge', 'cla__penalty': 'l2', 'rep__n_components': 0.9} con puntaje 0.8327
Entrenando modelo: LDA_PCA (3/8)...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Mejor modelo para LDA_PCA: {'cla__shrinkage': 'auto', 'cla__solver': 'lsqr', 'rep__n_components': 0.9} con puntaje 0.8133
Entrenando modelo: QDA_PCA (4/8)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Mejor modelo para QDA_PCA: {'cla__reg_param': 0.1, 'rep__n_components': 0.8} con puntaje 0.6803
Entrenando modelo: KNEIGHBORS_PCA (5/8)...
Fitting 5 folds for each of 192 candidates, totalling 960 fits
Mejor modelo para KNEIGHBORS_PCA: {'cla__algorithm': 'a